In [ ]:
import numpy as np
import math
import plotly.graph_objects as go
import plotly.express as px

In [ ]:
# deo koda za transformaciju koordinata (koordinatni pocetak preselimo u gornji-desni ugao, jer se dobijaju nesto precizniji rezultati)

def transform_pixel_array(points):
  return np.array([2000, 0, 0]) - np.array([1, -1, -1])*points

def transform_pixel(point):
  return np.array([2000 - point[0], point[1], point[2]])

In [ ]:
# deo koda za odredjivanje nevidljivih temena kutija sa slike

# 0   1   2   3   4   5   6
#p1, p2, p4, p5, p6, p7, p8
def nevidljivoTemeP3(pt):
    p58 = np.cross(pt[3], pt[6])
    p14 = np.cross(pt[0], pt[2])
    p = np.cross (p14, p58)
    p2 = np.cross(pt[1], p)

    q26 = np.cross(pt[1], pt[4])
    q15 = np.cross(pt[0], pt[3])
    q = np.cross(q26, q15)
    q7 = np.cross(q, pt[5])

    p3h = np.cross(q7,p2)
    p3h = np.round(p3h[:-1] / p3h[-1])
    return np.append(p3h, 1.0)

# 0   1   2   3   4   5   6
#p1, p2, p3, p5, p6, p7, p8
def nevidljivoTemeP4(pt):
    p15 = np.cross(pt[0], pt[3])
    p26 = np.cross(pt[1], pt[4])
    p = np.cross(p26, p15)
    p8 = np.cross(pt[6], p)

    q21 = np.cross(pt[1], pt[0])
    q65 = np.cross(pt[4], pt[3])
    q = np.cross(q65, q21)
    q3 = np.cross(q, pt[2])

    p4h = np.cross(q3, p8)
    p4h = np.round(p4h[:-1] / p4h[-1])
    return np.append(p4h, 1.0)


In [ ]:
# leva fotografija

# prvi (donji) kvadar
p1l = np.array([832, 1209, 1])
p2l = np.array([1380, 768, 1])
p4l = np.array([565, 928, 1])
p5l = np.array([821, 1134, 1])
p6l = np.array([1402, 692, 1])
p7l = np.array([1115, 524, 1])
p8l = np.array([545, 846, 1])
# p3l = ?, odredicemo preko osmoteme, ali necemo koristiti za F
p3l = nevidljivoTemeP3([p1l, p2l, p4l, p5l, p6l, p7l, p8l])

# drugi (gornji) kvadar
r1l = np.array([1015, 964, 1])
r2l = np.array([1158, 665, 1])
r4l = np.array([789, 890, 1])
r5l = np.array([1022, 855, 1])
r6l = np.array([1170, 571, 1])
r7l = np.array([969, 522, 1])
r8l = np.array([780, 786, 1])
# r3l = ?, necemo koristiti za F
r3l = nevidljivoTemeP3([r1l, r2l, r4l, r5l, r6l, r7l, r8l])

# desna fotografija

# prvi (donji) kvadar
p1r = np.array([614, 736, 1])
p2r = np.array([1092, 1153, 1])
p3r = np.array([1340, 917, 1])
p5r = np.array([602, 667, 1])
p6r = np.array([1104, 1090, 1])
p7r = np.array([1361, 850, 1])
p8r = np.array([862, 527, 1])
# p4r = ?, necemo koristiti za F
p4r = nevidljivoTemeP4([p1r, p2r, p3r, p5r, p6r, p7r, p8r])

# desni (gornji) kvadar
r1r = np.array([722, 747, 1])
r2r = np.array([1092, 857, 1])
r3r = np.array([1151, 729, 1])
r5r = np.array([714, 655, 1])
r6r = np.array([1107, 771, 1])
r7r = np.array([1163, 639, 1])
r8r = np.array([801, 545, 1])
# r4r = ?, necemo koristiti za F
r4r = nevidljivoTemeP4([r1r, r2r, r3r, r5r, r6r, r7r, r8r])

In [ ]:
# rezultati nevidljivih temena sa leve slike
print(f'P3L = {p3l}')
print(f'R3L = {r3l}')

P3L = [1.115e+03 5.240e+02 1.000e+00]
R3L = [966. 626.   1.]


In [ ]:
# rezultati nevidljivih temena sa desne slike
print(f'P4R = {p4r}')
print(f'R4R = {r4r}')

P4R = [863. 580.   1.]
R4R = [803. 627.   1.]


In [ ]:
# deo za odredjivanje fundamentalne F matrice
# koristimo po 12 tacaka sa leve i desne slike (necemo koristiti P3,R3 i P4,R4 koji su nevidljiva temena redom na levoj i desnoj slici)

leve_tacke = transform_pixel_array([p1l, p2l, p5l, p6l, p7l, p8l, r1l, r2l, r5l, r6l, r7l, r8l])
desne_tacke = transform_pixel_array([p1r, p2r, p5r, p6r, p7r, p8r, r1r, r2r, r5r, r6r, r7r, r8r])

def par_tacaka_jednacina(x, y):
  return np.array([x[0]*y[0], x[1]*y[0], x[2]*y[0], x[0]*y[1], x[1]*y[1], x[2]*y[1], x[0]*y[2], x[1]*y[2], x[2]*y[2]])

def dlt(pts1, pts2):
  jednacine = []
  for i in range(len(pts1)):
    jednacina = par_tacaka_jednacina(pts1[i], pts2[i])
    jednacine.append(jednacina)

  jednacine = np.array(jednacine)
  U, S, Vh = np.linalg.svd(jednacine)
  matP = Vh[-1].reshape(3,3)
  return matP/matP[2,2]

def F_matrix(left_points, right_points):
  return dlt(left_points, right_points)

In [ ]:
F = F_matrix(leve_tacke, desne_tacke)
print('Fundamentalna matrica:')
print(F)

Fundamentalna matrica:
[[ 1.25701772e-07 -6.30495033e-07 -5.16206749e-04]
 [-4.98743316e-07  1.51325492e-07 -7.93307934e-04]
 [-4.60071121e-04  1.64166883e-03  1.00000000e+00]]


In [ ]:
# test fundamentalne matrice
print('Sve provere parova tacaka sa leve i desne kamere treba da budu blizu nule')
for i in range(len(leve_tacke)):
  check = np.transpose(desne_tacke[i]) @ F @ leve_tacke[i]
  print(f'{i} : {check}')

print('Determinanta fundamentalne matrice treba da bude bliska nuli')
print(np.linalg.det(F))

Sve provere parova tacaka sa leve i desne kamere treba da budu blizu nule
0 : 0.0009746431142514389
1 : 0.0007186145269731936
2 : -0.0016708149539642614
3 : -0.0008387514224315051
4 : 0.0010981865531898283
5 : 9.806027521364058e-05
6 : 0.0005556315998240802
7 : -0.0025017491461940544
8 : 0.00032271709241193935
9 : 0.002057291718134624
10 : -0.001469911020171799
11 : 0.0006757800678005577
Determinanta fundamentalne matrice treba da bude bliska nuli
2.4874138013693324e-14


In [ ]:
# deo za odredjivanje osnovne matrice E

# matrica kalibracije kamere (rezolucija slike je 2000x1500, a na osnovu prethodnog domaceg, zizna daljina je priblizno 1500)
K = np.array([
    [1500,0,1000],
    [0,1500,750],
    [0,0,1]
])

def E_matrix(F, K):
  return np.transpose(K) @ F @ K

E = E_matrix(F, K)
print('Osnovna matrica E:')
print(E)

Osnovna matrica E:
[[ 0.28282899 -1.41861382 -1.29506438]
 [-1.12217246  0.34048236 -1.7678357 ]
 [-1.06264025  1.68700188  0.0238864 ]]


In [ ]:
# deo za odredjivanje matrica EC i AA, odnosno dekompozicije E = EC * AA

def svd(EE):
  U, SS, V = np.linalg.svd(EE)
  detU = np.linalg.det(U)
  detV = np.linalg.det(V)

  if np.sign(detU) != np.sign(detV):
    return np.linalg.svd(-EE)

  if detU == -1 and detV == -1:
    return -U, SS, -V

  return U, SS, V

def dekompozicija_E(EE):
  Q0 = np.array([
      [0, -1, 0],
      [1, 0, 0],
      [0, 0, 1]
      ])

  E0 = np.array([
      [0, 1, 0],
      [-1, 0, 0],
      [0, 0, 0]
  ])

  U, SS, V = svd(EE)
  EC = U @ E0 @ np.transpose(U)
  AA = U @ np.transpose(Q0) @ V

  return EC, AA

EC, AA = dekompozicija_E(E)
print('EC matrica:')
print(EC)
print('AA matrica:')
print(AA)

EC matrica:
[[ 1.32532605e-17  6.29404153e-01  4.50690699e-01]
 [-6.29404153e-01 -5.10941103e-18  6.33031047e-01]
 [-4.50690699e-01 -6.33031047e-01  1.23095472e-17]]
AA matrica:
[[ 0.00711657  0.69414948 -0.7197957 ]
 [-0.67269146  0.53591905  0.51017347]
 [ 0.73988888  0.48056974  0.47076233]]


In [ ]:
#test E = EC * AA
print('TEST: treba da vazi priblizno E = EC * AA')
print('E matrica:')
print(E)
print('EC * AA matrica:')
EE = EC @ AA
EE = (E[0, 0] / EE[0, 0]) * EE
print(EE)

TEST: treba da vazi priblizno E = EC * AA
E matrica:
[[ 0.28282899 -1.41861382 -1.29506438]
 [-1.12217246  0.34048236 -1.7678357 ]
 [-1.06264025  1.68700188  0.0238864 ]]
EC * AA matrica:
[[ 0.28282899 -1.74193106 -1.67706998]
 [-1.45887944  0.41727562 -2.36194501]
 [-1.32910297  2.0507629  -0.00455872]]


In [ ]:
# deo koda za odredjivanje matrica leve kamere T1 i desne kamere T2 (koja se poklapa sa svetskim koordinatnim sistemom)
def koso2v(A):
  return np.array([A[2, 1], A[0, 2], A[1, 0]]).reshape(3, 1)

def right_camera_matrix(K):
  return np.hstack((K, np.zeros((3, 1))))

def left_camera_matrix(K, EC, AA):
  CC = koso2v(EC)
  CC1 = (-np.transpose(AA)) @ CC
  return np.hstack((K @ np.transpose(AA), K @ CC1))

T1 = left_camera_matrix(K, EC, AA)
T2 = right_camera_matrix(K)

print('Matrica leve kamere T1:')
print(T1)
print('Matrica desne kamere T2:')
print(T2)

Matrica leve kamere T1:
[[-7.09120850e+02 -4.98863709e+02  1.58059565e+03  7.70771187e+02]
 [ 5.01377440e+02  1.18650868e+03  1.07392635e+03  4.58572761e+02]
 [-7.19795704e-01  5.10173475e-01  4.70762329e-01 -3.89283704e-01]]
Matrica desne kamere T2:
[[1.5e+03 0.0e+00 1.0e+03 0.0e+00]
 [0.0e+00 1.5e+03 7.5e+02 0.0e+00]
 [0.0e+00 0.0e+00 1.0e+00 0.0e+00]]


In [ ]:
# deo koda vezan za triangulaciju

def triangulacija_jednacina(T1, T2, m1, m2):
  return np.array([m1[1]*T1[2] - m1[2]*T1[1], -m1[0]*T1[2] + m1[2]*T1[0], m2[1]*T2[2] - m2[2]*T2[1], -m2[0]*T2[2] + m2[2]*T2[0]])

def triangulacija_tacke(T1, T2, m1, m2):
  jedM = triangulacija_jednacina(T1, T2, m1, m2)
  _, _, V = np.linalg.svd(jedM)
  M = V[-1] / V[3, 3]
  return M

# provera triangulacije za tacku P1
P1 = triangulacija_tacke(T1, T2, transform_pixel(p1l), transform_pixel(p1r))
print(P1)
print((T1 @ P1) / (T1 @ P1)[-1])
print(transform_pixel(p1l))
print((T2 @ P1) / (T2 @ P1)[-1])
print(transform_pixel(p1r))

[-0.2815843   0.01879428 -1.12308582  1.        ]
[1.15353593e+03 1.22771177e+03 1.00000000e+00]
[1168 1209    1]
[1.37608564e+03 7.24898255e+02 1.00000000e+00]
[1386  736    1]


In [ ]:
# deo koda za prikaz rekonstruisane 3D scene

#                               0    1    2    3    4   5     6   7     8   9    10   11   12   13   14   15
leve = transform_pixel_array([p1l, p2l, p3l, p4l, p5l, p6l, p7l, p8l, r1l, r2l, r3l, r4l, r5l, r6l, r7l, r8l])
desne = transform_pixel_array([p1r, p2r, p3r, p4r, p5r, p6r, p7r, p8r, r1r, r2r, r3r, r4r, r5r, r6r, r7r, r8r])

tacke3D = []
for i in range(len(leve)):
  tacke3D.append(triangulacija_tacke(T1, T2, leve[i], desne[i]))

for i in range(len(tacke3D)):
  tacke3D[i] = tacke3D[i][:-1]

ivice3D = np.array([
    [0, 1], [1, 2], [2, 3], [3, 0],
    [0, 4], [1, 5], [2, 6], [3, 7],
    [4, 5], [5, 6], [6, 7], [7, 4],

    [8, 9], [9, 10], [10, 11], [11, 8],
    [8, 12], [9, 13], [10, 14], [11, 15],
    [12, 13], [13, 14], [14, 15], [15, 12]
])

data = np.transpose(tacke3D)
xdata = data[0]
ydata = data[1]
zdata = data[2]

# u data1 ubacujemo sve sto treba nacrtati
data1 = []
for i in range(len(ivice3D)):
  data1.append(go.Scatter3d(x=[xdata[ivice3D[i][0]], xdata[ivice3D[i][1]]], y=[ydata[ivice3D[i][0]], ydata[ivice3D[i][1]]],z=[zdata[ivice3D[i][0]], zdata[ivice3D[i][1]]]))

fig = go.Figure(data=data1)
fig.update_layout(showlegend = False)
fig.show()

In [ ]:
# prikaz svih trazenih rezultata u okviru jedne celije
# F, E, EC, A, T1, T2, 3D koordinate

print('Fundamentalna matrica F:')
print(F)
print()

print('Osnovna matrica E:')
print(E)
print()

print('matrica Eo (EC):')
print(EC)
print()

print('matrica A:')
print(AA)
print()

print('matrica leve kamere T1:')
print(T1)
print()

print('matrica desne kamere T2:')
print(T2)
print()

print('rekonstruisane 3D koordinate tacaka iz triangulacije: ')
broj_temena = len(tacke3D) // 2

for i in range(broj_temena):
  print(f'P{i + 1} = {tacke3D[i]}')

for i in range(broj_temena, len(tacke3D)):
  print(f'R{i + 1 - broj_temena} = {tacke3D[i]}')

Fundamentalna matrica F:
[[ 1.25701772e-07 -6.30495033e-07 -5.16206749e-04]
 [-4.98743316e-07  1.51325492e-07 -7.93307934e-04]
 [-4.60071121e-04  1.64166883e-03  1.00000000e+00]]

Osnovna matrica E:
[[ 0.28282899 -1.41861382 -1.29506438]
 [-1.12217246  0.34048236 -1.7678357 ]
 [-1.06264025  1.68700188  0.0238864 ]]

matrica Eo (EC):
[[ 1.32532605e-17  6.29404153e-01  4.50690699e-01]
 [-6.29404153e-01 -5.10941103e-18  6.33031047e-01]
 [-4.50690699e-01 -6.33031047e-01  1.23095472e-17]]

matrica A:
[[ 0.00711657  0.69414948 -0.7197957 ]
 [-0.67269146  0.53591905  0.51017347]
 [ 0.73988888  0.48056974  0.47076233]]

matrica leve kamere T1:
[[-7.09120850e+02 -4.98863709e+02  1.58059565e+03  7.70771187e+02]
 [ 5.01377440e+02  1.18650868e+03  1.07392635e+03  4.58572761e+02]
 [-7.19795704e-01  5.10173475e-01  4.70762329e-01 -3.89283704e-01]]

matrica desne kamere T2:
[[1.5e+03 0.0e+00 1.0e+03 0.0e+00]
 [0.0e+00 1.5e+03 7.5e+02 0.0e+00]
 [0.0e+00 0.0e+00 1.0e+00 0.0e+00]]

rekonstruisane 3D koo